In [60]:
import torch
import os
import sys
import subprocess
import copy
import torch.nn.functional as F
import torch.optim as optim
import datetime

from torch_geometric.loader import DataLoader as PyGDataLoader
from models.complex_models.tu_models import SparseCIN
from models.complex_models.molecular_models import EmbedSparseCIN
from lib.helpers.data_helpers import get_dataloaders, load_graph_dataset, load_dataset, DataLoader
from lib.helpers.lr_scheduler import get_lr_scheduler
from lib.utils.random_seed import set_random_seed
from lib.helpers.trainer import Trainer
from lib.helpers.eval_helpers import extract_results_molecular_datasets, extract_results_tu_datasets, extract_results_sr_datasets
from tools.parser import get_parser, validate_args
from lib.helpers.model_helpers import get_complex_model, get_graph_model
from lib.utils.sr_utils import sr_families
from lib.layers.cin_conv import SparseCINConv


In [61]:
batch_size = 128
complex_type = 'path'
conv_type = 'B'
dataset_name = 'NCI1'
device = 0
drop_position = 'lin2'
drop_rate = 0
emb_dim = 2
epochs = 150
eval_metric = 'accuracy'
exp_name = 'pcn-nci1'
final_readout = 'sum'
folds = 10
graph_norm = 'bn'
init_method = 'sum'
jump_mode = None
lr = 0.0005
lr_scheduler = 'StepLR'
lr_scheduler_decay_rate = 0.2
lr_scheduler_decay_steps = 60
max_dim = 7
model = 'sparse_cin'
nonlinearity = 'relu'
num_fc_layers = 2
num_layers = 2
num_layers_combine = 1
num_layers_update = 2
preproc_jobs = 32
readout = 'sum'
task_type = 'classification'
train_eval_period = 50

fold = 1
max_ring_size = None
seed = 1234
flow_points=400
flow_classes=3
use_edge_features = True
simple_features = False
include_down_adj=False
train_orient = 'default'
test_orient = 'default'
num_workers = 0


In [62]:
_, _, _, _, num_classes, _, num_node_labels = load_graph_dataset(dataset_name, seed=1, fold=fold, 
                                                                 max_ring_size=max_ring_size)
num_classes, num_node_labels


(2, 37)

In [63]:
dataset = load_dataset(dataset_name, 
                        complex_type = complex_type, 
                        max_dim=max_dim,
                        seed=seed, 
                        fold=fold, 
                        init_method=init_method, 
                        flow_points=flow_points, 
                        flow_classes=flow_classes,
                        num_classes=num_classes,
                        num_node_labels=num_node_labels,
                        max_ring_size=max_ring_size,
                        use_edge_features=use_edge_features,
                        simple_features=simple_features,
                        include_down_adj=include_down_adj,
                        n_jobs=preproc_jobs,
                        train_orient=train_orient, 
                        test_orient=test_orient,
                        )


In [65]:
dataset[0].nodes.x.size()

torch.Size([21, 37])

In [66]:
# Get dataset split
split_idx = dataset.get_idx_split()

# Instantiate data loaders
train_loader = DataLoader(dataset.get_split('train'), batch_size=batch_size,
    shuffle=True, num_workers=num_workers, max_dim=dataset.max_dim)
valid_loader = DataLoader(dataset.get_split('valid'), batch_size=batch_size,
    shuffle=False, num_workers=num_workers, max_dim=dataset.max_dim)
test_split = split_idx.get("test", None)
test_loader = None
if test_split is not None:
    test_loader = DataLoader(dataset.get_split('test'), batch_size=batch_size,
        shuffle=False, num_workers=num_workers, max_dim=dataset.max_dim)

train_loader, valid_loader, test_loader, dataset


(<lib.helpers.data_helpers.DataLoader at 0x7f3aaf81bc70>,
 None,
 TUDataset(4110))

In [67]:
use_coboundaries = False
use_boundaries = False
disable_graph_norm = False
readout_dims = None
in_dropout_rate = 0.0

model = SparseCIN(dataset.num_features_in_dim(0),             # num_input_features
                dataset.num_node_labels,
                dataset.num_classes,                          # number of classes
                num_layers,                              # num_layers
                emb_dim,                                 # hidden
                dropout_rate=drop_rate,                  # dropout_rate
                in_dropout_rate=in_dropout_rate,             # dropout rate for input
                max_dim=dataset.max_dim,                      # max_dim
                jump_mode=jump_mode,                     # jumping knowledge
                nonlinearity=nonlinearity,               # nonlinearity
                readout=readout,                         # readout
                final_readout=final_readout,             # final readout
                apply_dropout_before=drop_position,      # where to apply dropout
                use_coboundaries=use_coboundaries,            # whether to use coboundaries
                use_boundaries=use_boundaries,
                include_down_adj=include_down_adj,
                init_reduce=init_method,
                graph_norm=graph_norm,                   # normalization layer
                disable_graph_norm=disable_graph_norm,
                # readout_dims=readout_dims,               # readout_dims
                num_layers_update=num_layers_update,     # number of update layers
                num_layers_combine=num_layers_combine,   # number of combine layers
                num_fc_layers=num_fc_layers,             # number of fc layers after msg prop
                conv_type=conv_type,                     # conv type A or B
                )

In [68]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
device

device(type='cpu')

In [69]:
batch = next(iter(train_loader))
num_samples = batch.cochains[0].x.size(0)
num_samples

3696

In [80]:
batch.cochains

{0: <lib.data.cochain.CochainBatch at 0x7f3a88c656d0>,
 1: <lib.data.cochain.CochainBatch at 0x7f3a88c65730>,
 2: <lib.data.cochain.CochainBatch at 0x7f3a88c65760>,
 3: <lib.data.cochain.CochainBatch at 0x7f3a88c659a0>,
 4: <lib.data.cochain.CochainBatch at 0x7f3a88c659d0>,
 5: <lib.data.cochain.CochainBatch at 0x7f3a88c65a00>,
 6: <lib.data.cochain.CochainBatch at 0x7f3a88c65a30>,
 7: <lib.data.cochain.CochainBatch at 0x7f3a88c65a60>}

In [72]:
batch.num_complexes

128

In [14]:
batch.dimension

7

In [13]:
cochain_params = batch.get_all_cochain_params(max_dim=batch.dimension)
cochain_params

In [73]:
cinconv = model.convs[0]
emb = cinconv(cochain_params[0])[0]
emb_up, emb_bound = emb[:, 0], emb[:, 1]

emb    

tensor([[0.0197, 0.0000],
        [0.0197, 0.0000],
        [0.0940, 0.1545],
        ...,
        [0.2773, 0.0000],
        [0.2773, 0.0000],
        [0.0000, 0.0000]], grad_fn=<ReluBackward0>)

In [51]:
# emb0, emb1 = cinconv(cochain_params[0])[0], cinconv(cochain_params[1])[0]
# F.cosine_similarity(emb0, emb1)

RuntimeError: The size of tensor a (4084) must match the size of tensor b (4408) at non-singleton dimension 0

In [74]:
def euclidean_dist(x, y, squared=True):
    m, n = x.size(0), y.size(0)
    mat1 = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(m, n)
    mat2 = torch.pow(y, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    distmat = mat1 + mat2
    distmat.addmm_(x, y.t(), beta=1,alpha=-2)
    return distmat

euclidean_dist(emb0, emb1).shape

torch.Size([4084, 4408])

In [91]:
batch.cochains[0].x.shape, batch.cochains[1].x.shape

(torch.Size([3696, 37]), torch.Size([4012, 37]))

In [90]:
d = euclidean_dist(batch.cochains[0].x, batch.cochains[1].x)
d.shape

torch.Size([3696, 4012])

In [57]:
x, y = emb0, emb1
m, n = x.size(0), y.size(0)
mat1 = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(m, n)
mat1

tensor([[0.1367, 0.1367, 0.1367,  ..., 0.1367, 0.1367, 0.1367],
        [0.1367, 0.1367, 0.1367,  ..., 0.1367, 0.1367, 0.1367],
        [0.1102, 0.1102, 0.1102,  ..., 0.1102, 0.1102, 0.1102],
        ...,
        [1.0160, 1.0160, 1.0160,  ..., 1.0160, 1.0160, 1.0160],
        [1.0160, 1.0160, 1.0160,  ..., 1.0160, 1.0160, 1.0160],
        [0.3118, 0.3118, 0.3118,  ..., 0.3118, 0.3118, 0.3118]],
       grad_fn=<ExpandBackward0>)

In [33]:
for k in range(len(cochain_params)): # n = len(cochain_params)
    # for each cochain, choose 1 positive (k-1) and n-2 negatives (not k and k-1)

    anchor = cochain_params[k]
    pos = cochain_params[k-1] if k>0 else cochain_params[k]

    negs = []
    if k > 0:
        negs += cochain_params[:k-1]
    if k+1 < len(cochain_params): 
        negs += cochain_params[k+1:]
    if k == 0:
        negs = negs[1:] # drop 1 negative of this k to have same number of negatives as other k.
    
    
    emb_anchor = cinconv(anchor)
    emb_pos = cinconv(pos)
    emb_negs = []
    for neg in negs:
        emb_negs.append(cinconv(neg))
    
     
    
    


0 6
1 6
2 6
3 6
4 6
5 6
6 6
7 6


In [46]:
emb0, emb1 = cinconv(cochain_params[0])[0], cinconv(cochain_params[1])[0]
F.cosine_similarity(emb0, emb1)

RuntimeError: The size of tensor a (4408) must match the size of tensor b (4084) at non-singleton dimension 0

In [50]:
emb0.shape

torch.Size([4084, 2])